# Sleep Learning Engine - Drive-backed low-RAM cloud render

This notebook is the **personal / recurring-render** variant of
the public Colab notebook at `docs/cloud/low_ram_render.ipynb`.
Use it when you have a stable ambient library and a recurring
render workflow:

- The 97 normalised ambient mp3 files live in Drive and are
  mounted on every Colab session (cell 2). You upload them
  **once** and forget about them.
- The script and the background image or video change with
  every project, so you upload them fresh at the start of
  each render (cell 3).

**Recommended Drive layout** (create it once, reuse forever):

```
My Drive/
└── sleep-learning-engine/
    └── ambient/      <- 97 normalised mp3 tracks (persistent)
```

The script and background image are uploaded per-session and
are NOT stored in Drive (they change with every project).

**Runtime setup (one-time, every fresh Colab session):**
1. Runtime -> Change runtime type -> T4 GPU -> RAM amplia = On -> Save
2. Reconnect when prompted
3. Click Runtime -> Run all (Ctrl+F9)

**Cost:** free. Colab free sessions cap at 12 hours. A 6-minute
video finishes in well under 10 minutes end-to-end.

In [1]:
# 1. Install sleep_learning_engine from the public repo. Tarball URL
# (not a git clone) so pip does not need git credentials.
!pip install -q https://github.com/fernandojjq/sleep-learning-engine/archive/refs/heads/main.tar.gz

# Verify the runtime actually has a GPU. The previous version only
# checked the ffmpeg encoder list, which lies: h264_nvenc can be in
# the list even when no GPU is bound to the container. The real test
# is nvidia-smi (raises if no GPU is present).
import subprocess
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"], capture_output=True, text=True, timeout=15
)
if smi.returncode == 0 and smi.stdout.strip():
    print("GPU detected:")
    print(smi.stdout)
    print("Note: NVENC does NOT use VRAM. 0.0/15.0 GB USED is normal -",
          "NVENC runs on dedicated T4 silicon, not on the GPU cores.",
          "If VRAM shows >0 GB, the encode is going through the GPU",
          "and is therefore hardware-accelerated.")
else:
    print("=" * 60)
    print("NO GPU DETECTED in this Colab runtime.")
    print("Encode will fall back to libx264 (CPU), 5-10x slower.")
    print("")
    print("Fix: Runtime -> Change runtime type -> T4 GPU -> Save.")
    print("Reconnect when prompted. Re-run this cell.")
    print("=" * 60)

nvenc = subprocess.run(
    ["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True
).stdout
if "h264_nvenc" in nvenc:
    print("NVENC encoder: AVAILABLE")
else:
    print("NVENC encoder: NOT in ffmpeg build (libx264 fallback)")

     - 371.9 kB 11.4 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.1/296.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.6/475.6 kB 32.7 MB/s eta 0:00:00
GPU detected:
Tesla T4, 15360 MiB, 580.82.07

Note: NVENC does NOT use VRAM. 0.0/15.0 GB USED is normal - NVENC runs on dedicated T4 silicon, not on the GPU cores. If VRAM shows >0 GB, the encode is going through the GPU and is therefore hardware-accelerated.
NVENC encoder: AVAILABLE


In [2]:
# 2. Mount Google Drive and copy the persistent ambient library
# into the writable working directory. This is a one-time copy per
# session; subsequent cells skip files that already exist.
from google.colab import drive
import os, shutil, glob

# Path conventions (edit if you put things elsewhere):
DRIVE_ROOT = "/content/drive/MyDrive/sleep-learning-engine"
AMBIENT_SRC = f"{DRIVE_ROOT}/ambient"
WORK = "/content/working"
ASSETS = f"{WORK}/assets"
AMBIENT_DST = f"{ASSETS}/ambient"

# Mount. force_remount=False avoids the OAuth popup when the
# session was started with Drive already attached. The first
# time in a session this WILL pop up a Google login.
drive.mount('/content/drive', force_remount=False)

for d in (ASSETS, AMBIENT_DST, f"{WORK}/output"):
    os.makedirs(d, exist_ok=True)

if not os.path.exists(AMBIENT_SRC):
    print(f"WARNING: {AMBIENT_SRC} not found.")
    print("Create the folder in Drive, upload the mp3 files, then re-run this cell.")
else:
    copied = 0
    for ext in ("mp3", "ogg", "wav", "flac", "m4a", "aac"):
        for src in glob.glob(f"{AMBIENT_SRC}/*.{ext}"):
            dst = os.path.join(AMBIENT_DST, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                copied += 1
    total = len(glob.glob(f"{AMBIENT_DST}/*"))
    print(f"Ambient library: {total} tracks ({copied} new this session)")

Mounted at /content/drive


KeyboardInterrupt: 

In [ ]:
# 3. Upload the script and the background image (or short video) for
# THIS render. The script and background change with every project,
# so the only thing that is persistent across renders is the ambient
# library (handled in cell 2 via Drive).
from google.colab import files
import os, glob, shutil
from PIL import Image  # <-- Línea corregida para poder validar el tamaño

print("Upload the script text file (.txt) and the background image or video for THIS render...")
uploaded = files.upload()
for name, data in uploaded.items():
    target = f"{ASSETS}/{name}"
    with open(target, "wb") as fh:
        fh.write(data)
print(f"Uploaded: {sorted(uploaded.keys())}")

# Resolve the paths. First .txt is the script, first video is the
# loopable background, first image is the still background.
SCRIPT = ""
for pat in ("*.txt",):
    matches = sorted(glob.glob(f"{ASSETS}/{pat}"))
    if matches:
        SCRIPT = matches[0]
        break

BG_IMAGE = ""
BG_VIDEO = ""
for pat in ("*.mp4", "*.mov", "*.webm"):
    matches = sorted(glob.glob(f"{ASSETS}/{pat}"))
    if matches:
        BG_VIDEO = matches[0]
        break
if not BG_VIDEO:
    for pat in ("*.jpg", "*.jpeg", "*.png"):
        matches = sorted(glob.glob(f"{ASSETS}/{pat}"))
        if matches:
            BG_IMAGE = matches[0]
            break

# ------ CORRECCIÓN CRÍTICA: Validar y adaptar dimensiones para NVENC ------
if BG_IMAGE:
    try:
        with Image.open(BG_IMAGE) as img:
            w, h = img.size
            # Si las dimensiones son impares o inferiores al mínimo de la GPU, reescalamos
            if w < 1280 or h < 720 or w % 2 != 0 or h % 2 != 0:
                print(f"⚠️ Imagen incompatible con GPU ({w}x{h}). Reescalando automáticamente a Full HD (1920x1080)...")
                img_rgb = img.convert("RGB")
                img_resized = img_rgb.resize((1920, 1080), Image.Resampling.LANCZOS)
                img_resized.save(BG_IMAGE, format="PNG")
                print("✅ Imagen corregida con éxito.")
    except Exception as e:
        print(f"Advertencia al validar la imagen de fondo: {e}")
# --------------------------------------------------------------------------

print(f"Script:      {SCRIPT or '<missing>'}")
print(f"Background:  {BG_IMAGE or BG_VIDEO or '<missing>'}")
print(f"  is video:  {bool(BG_VIDEO)}")

if not SCRIPT:
    raise SystemExit("Need a .txt script. Upload one in this cell.")
if not (BG_IMAGE or BG_VIDEO):
    raise SystemExit("Need a background image or video. Upload one in this cell.")

In [ ]:
# 4. Render acelerado por GPU con blindaje absoluto de imágenes y progreso en vivo
import os
import subprocess
import sys
from PIL import Image

# 1. Auto-resolver rutas críticas por si acaso se reinició la memoria de Colab
WORK = "/content/working"
if 'SCRIPT' not in locals() or not SCRIPT:
    import glob
    matches = sorted(glob.glob(f"{WORK}/assets/*.txt"))
    SCRIPT = matches[0] if matches else ""

if 'BG_IMAGE' not in locals() or not BG_IMAGE:
    import glob
    matches = (sorted(glob.glob(f"{WORK}/assets/*.png")) +
               sorted(glob.glob(f"{WORK}/assets/*.jpg")) +
               sorted(glob.glob(f"{WORK}/assets/*.jpeg")))
    BG_IMAGE = matches[0] if matches else ""

if 'BG_VIDEO' not in locals() or not BG_VIDEO:
    import glob
    matches = sorted(glob.glob(f"{WORK}/assets/*.mp4"))
    BG_VIDEO = matches[0] if matches else ""

# 2. BLINDAJE ABSOLUTO: Forzar dimensiones válidas para la GPU T4 antes del comando
if BG_IMAGE and os.path.exists(BG_IMAGE):
    print(f"🔍 Analizando fondo: {BG_IMAGE}")
    try:
        with Image.open(BG_IMAGE) as img:
            w, h = img.size
            print(f"   -> Dimensiones actuales encontradas: {w}x{h}")
            if w < 1280 or h < 720 or w % 2 != 0 or h % 2 != 0:
                print(f"   ⚠️ Dimensiones inválidas para NVENC. Reescalando a 1920x1080...")
                img_rgb = img.convert("RGB")
                img_resized = img_rgb.resize((1920, 1080), Image.Resampling.LANCZOS)
                img_resized.save(BG_IMAGE, format="PNG")
                print("   ✅ ¡Imagen reescalada con éxito!")
    except Exception as e:
        print(f"   ❌ Error crítico: La imagen no se puede leer o está corrupta ({e}).")
        print("   Reparando entorno: Generando un fondo negro Full HD de reemplazo...")
        img_fallback = Image.new("RGB", (1920, 1080), color="black")
        img_fallback.save(BG_IMAGE, format="PNG")
        print("   ✅ Fondo de emergencia generado correctamente.")

# 3. Configurar comando final
OUT_STEM = "sleep_learning_engine-drive"
cmd = [
    "python", "-m", "sleep_learning_engine", "render",
    "--script", SCRIPT,
    "--output-stem", OUT_STEM,
]
if os.path.exists(BG_IMAGE):
    cmd += ["--background-image", BG_IMAGE]
elif os.path.exists(BG_VIDEO):
    cmd += ["--background-video", BG_VIDEO]

# 4. Enlazar controladores NVIDIA de tu entorno Pro
env = {**os.environ, "TMP": WORK, "TEMP": WORK}
colab_nvidia_path = "/usr/lib64-nvidia"
if os.path.exists(colab_nvidia_path):
    env["LD_LIBRARY_PATH"] = colab_nvidia_path + ":" + env.get("LD_LIBRARY_PATH", "")

print("\n" + "="*60)
print("EJECUTANDO MOTOR DE RENDERIZADO (PROGRESO EN VIVO):")
print("="*60)

# 5. Ejecución con streaming en tiempo real
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    cwd=WORK
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
print("="*60)
if proc.returncode != 0:
    raise SystemExit(f"❌ El renderizado falló con código de salida {proc.returncode}")
print("🎉 ¡Video renderizado exitosamente en hardware NVENC!")

In [ ]:
# 5. Locate the final MP4 and download it to your machine.
import glob, os
from google.colab import files

candidates = sorted(glob.glob(f"{WORK}/output/{OUT_STEM}.mp4"))
if not candidates:
    raise SystemExit("No MP4 was produced. Check the render cell for errors.")
output = candidates[0]
print(f"Final video: {output}")
print(f"Size: {os.path.getsize(output)/1e6:.1f} MB")
files.download(output)

## Why this notebook and not the public one

The public `docs/cloud/low_ram_render.ipynb` asks you to re-upload
all 96 ambient mp3 files at the start of every session. If you
render more than once a week, that 5-10 min upload is friction
this notebook removes by mounting Drive and pulling the library
from there.

Both notebooks run the same `sleep_learning_engine` pipeline and
produce identical MP4s. Use whichever matches your workflow.

## When to use Kaggle instead

If you find yourself running this notebook many times per week,
and you want guaranteed T4 GPU access (Colab free is best-effort,
can be 0/15 GB during peak hours), the Kaggle notebook at
`docs/cloud/kaggle_render.ipynb` is the right answer. Kaggle
gives 2x T4 (30 GB VRAM) and 30 GPU hours/week. The dataset path
is different: you upload the mp3 files as a Kaggle dataset at
`/kaggle/input/<slug>/` instead of Google Drive.